# 4.4 GMM과 EM 알고리즘 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter04_4_gmm_em.ipynb)

책 본문: [Chapter 4.4](https://smhanlab.com/book-ml/kor/ml1/chapter04/4.html)

numpy/matplotlib만으로 본문의 코드를 그대로 실행해, 본문이 주장하는 현상들이
실제로 일어나는지 확인합니다:

1. 6개 데이터(1차원)에서 본문의 `em_step`을 돌리고 우도의 단조증가·수렴 확인
2. 3개 블롭 2차원 데이터에서 2D GMM을 EM으로 학습
3. **책임값 기반 색칠 + likelihood 등고선** 그림 (→ `ch15_gmm_contour.svg`, 본문 그림)
4. \\(K=1\sim4\\) **BIC 곡선** (→ `ch15_gmm_bic.svg`, 본문 그림)
5. 무작위 초기화 30회 — **지역최적해·분산 붕괴** 실패 사례 확인
6. GMM 잠재변수 다이어그램 (→ `ch15_gmm_latent_var.svg`, 본문 그림)

> **주의**: 본문에 적힌 특정 숫자(log-likelihood 표, BIC 값, 실패 횟수)는
> 정확한 재현을 목표로 하지 않습니다 — "같은 현상이 실제로 일어나는가"를
> 확인하는 것이 이 노트북의 목적입니다.


## 1. 1차원: 6개 데이터에서 EM (본문 코드 그대로)

본문의 작은 예제 — 데이터 \\([1.0, 1.5, 0.8, 9.2, 10.1, 9.8]\\), \\(K=2\\),
초기값 \\(\mu=[2,8],\ \sigma=[1,1],\ \pi=[0.5,0.5]\\). `em_step`은 본문에
있는 함수를 그대로 사용합니다. 매 반복마다 log-likelihood가 절대 줄지
않는지(옌센 부등식 섹션의 보장)를 표로 확인합니다.


In [1]:
import math
import numpy as np

def gaussian_pdf(x, mu, sigma):
    return (1 / (sigma * math.sqrt(2 * math.pi))) * math.exp(-((x - mu) ** 2) / (2 * sigma ** 2))

def em_step(data, means, sigmas, weights):
    K, n = len(means), len(data)
    # E-step: 책임값(responsibility) 계산
    resp = []
    for x in data:
        probs = [weights[k] * gaussian_pdf(x, means[k], sigmas[k]) for k in range(K)]
        total = sum(probs)
        resp.append([p / total for p in probs])
    # M-step: 책임값으로 가중평균한 파라미터 재추정
    Nk = [sum(resp[i][k] for i in range(n)) for k in range(K)]
    new_means = [sum(resp[i][k] * data[i] for i in range(n)) / Nk[k] for k in range(K)]
    new_sigmas = [math.sqrt(sum(resp[i][k] * (data[i] - new_means[k]) ** 2
                                  for i in range(n)) / Nk[k]) for k in range(K)]
    new_weights = [Nk[k] / n for k in range(K)]
    return new_means, new_sigmas, new_weights

# log-likelihood = sum_i log( sum_k pi_k N(x_i; mu_k, sigma_k^2) )
def gmm1d_loglik(data, means, sigmas, weights):
    return sum(math.log(sum(weights[k] * gaussian_pdf(x, means[k], sigmas[k])
                            for k in range(len(means)))) for x in data)

data = [1.0, 1.5, 0.8, 9.2, 10.1, 9.8]
means, sigmas, weights = [2.0, 8.0], [1.0, 1.0], [0.5, 0.5]

ll = gmm1d_loglik(data, means, sigmas, weights)
print(f"반복 0 (초기값)  logL = {ll:9.4f}   mu = {[round(m, 3) for m in means]}   sigma = {[round(s, 3) for s in sigmas]}   pi = {[round(w, 3) for w in weights]}")
for it in range(1, 51):
    means, sigmas, weights = em_step(data, means, sigmas, weights)
    ll_new = gmm1d_loglik(data, means, sigmas, weights)
    print(f"반복 {it:2d}          logL = {ll_new:9.4f}   mu = {[round(m, 3) for m in means]}   sigma = {[round(s, 3) for s in sigmas]}   pi = {[round(w, 3) for w in weights]}")
    if abs(ll_new - ll) < 1e-9:
        print(f"\n반복 {it}에서 수렴 (log-likelihood 더 이상 증가하지 않음)")
        ll = ll_new
        break
    ll = ll_new

# 수렴 후 책임값 — (1,0) 또는 (0,1)으로 딱 붙어야 한다
print("\n수렴 후 책임값 (행 = x = 1.0, 1.5, 0.8, 9.2, 10.1, 9.8):")
for x in data:
    p = [weights[k] * gaussian_pdf(x, means[k], sigmas[k]) for k in range(2)]
    t = sum(p)
    print(f"  x = {x:4.1f}   gamma = ({p[0] / t:.6f}, {p[1] / t:.6f})")
print("\n(참고: 본문 표의 숫자(초기 -15.56 → -6.05)는 재현 대상이 아님 —")
print(" 이번 실행의 실제 값이 위와 같이 출력됨. 단조증가·수렴이라는 '현상'이")
print(" 성립하는지가 확인 포인트다.)")


반복 0 (초기값)  logL =  -15.5625   mu = [2.0, 8.0]   sigma = [1.0, 1.0]   pi = [0.5, 0.5]
반복  1          logL =   -6.0548   mu = [1.1, 9.7]   sigma = [0.294, 0.374]   pi = [0.5, 0.5]
반복  2          logL =   -6.0548   mu = [1.1, 9.7]   sigma = [0.294, 0.374]   pi = [0.5, 0.5]

반복 2에서 수렴 (log-likelihood 더 이상 증가하지 않음)

수렴 후 책임값 (행 = x = 1.0, 1.5, 0.8, 9.2, 10.1, 9.8):
  x =  1.0   gamma = (1.000000, 0.000000)
  x =  1.5   gamma = (1.000000, 0.000000)
  x =  0.8   gamma = (1.000000, 0.000000)
  x =  9.2   gamma = (0.000000, 1.000000)
  x = 10.1   gamma = (0.000000, 1.000000)
  x =  9.8   gamma = (0.000000, 1.000000)

(참고: 본문 표의 숫자(초기 -15.56 → -6.05)는 재현 대상이 아님 —
 이번 실행의 실제 값이 위와 같이 출력됨. 단조증가·수렴이라는 '현상'이
 성립하는지가 확인 포인트다.)


## 2. 2차원 GMM: 데이터와 모델

3개 타원형 블롭(각 50점, seed 42) — **왼쪽은 세로로 길쭉한 타원, 위쪽은
원형, 아래쪽은 가로로 길쭉한 타원**. 2차원에서 \\(\sigma_k^2\\)는 스칼라가
아니라 공분산 **행렬** \\(\Sigma_k\\)가 되고, M-step은 가중평균(\\(\mu_k\\))
에 **가중 공분산**을 추가할 뿐이다(본문의 M-step 공식).


In [2]:
import numpy as np

rng = np.random.RandomState(42)

centers = np.array([[1.0, 1.5],
                    [4.0, 4.0],
                    [4.0, 1.0]])
covs_true = [np.array([[0.09, 0.0], [0.0, 0.36]]),  # 왼쪽: 세로로 길쭉한 타원
             np.eye(2) * 0.25,                       # 위: 원형
             np.array([[0.30, 0.0], [0.0, 0.09]])]   # 아래: 가로로 길쭉한 타원
X = np.vstack([rng.multivariate_normal(centers[k], covs_true[k], size=50) for k in range(3)])
print(f"데이터: {X.shape[0]}개 점, 전체 평균 = {X.mean(0).round(3)}")
print("(본문의 퇴화 해 중심이 (2.98, 2.04) 근처였음 — 이 데이터의 평균과 같은 위치.)")

def mvn_logpdf(X, mu, cov):
    # X의 각 행 x에 대해 log N(x; mu, cov) — 콜레스키 분해로 안정하게
    d = X.shape[1]
    L = np.linalg.cholesky(cov)
    z = np.linalg.solve(L, (X - mu).T)              # (d, n)
    maha = (z ** 2).sum(0)
    return -0.5 * (d * np.log(2 * np.pi) + 2 * np.log(np.diag(L)).sum() + maha)

# log prod_i sum_k pi_k N(x_i; mu_k, Sigma_k) — log-sum-exp으로 수치 안정
def gmm2d_loglik(X, mus, covs, pis):
    lle = np.stack([np.log(max(pis[k], 1e-300)) + mvn_logpdf(X, mus[k], covs[k]) for k in range(len(mus))])
    m = lle.max(0, keepdims=True)
    return float(np.sum(m + np.log(np.exp(lle - m).sum(0))))

# 본문의 em_step을 2차원으로: E-step은 log-space의 책임값, M-step은 가중평균 + 가중 공분산
def em_step_2d(X, mus, covs, pis, jitter=1e-6):
    n, d = X.shape
    K = len(mus)
    lle = np.stack([np.log(max(pis[k], 1e-300)) + mvn_logpdf(X, mus[k], covs[k]) for k in range(K)])
    m = lle.max(0, keepdims=True)
    G = np.exp(lle - m).T                            # (n, K) 책임값
    G /= G.sum(1, keepdims=True)
    Nk = G.sum(0)
    new_mus = (G.T @ X) / Nk[:, None]
    new_covs = []
    for k in range(K):
        diff = X - new_mus[k]
        # 가중 공분산: (G[:,k] * (x_i - mu_k))^T (x_i - mu_k) / N_k + 정규화
        new_covs.append((G[:, k, None] * diff).T @ diff / Nk[k] + jitter * np.eye(d))
    return new_mus, new_covs, Nk / n

def fit_gmm(X, mus, covs, pis, max_iter=200, tol=1e-9):
    # EM 반복 + 매 반복의 (반복수, logL, 파라미터 복사본) 기록
    ll = gmm2d_loglik(X, mus, covs, pis)
    hist = [(0, ll, mus.copy(), [c.copy() for c in covs], pis.copy())]
    for it in range(1, max_iter + 1):
        mus, covs, pis = em_step_2d(X, mus, covs, pis)
        ll_new = gmm2d_loglik(X, mus, covs, pis)
        hist.append((it, ll_new, mus.copy(), [c.copy() for c in covs], pis.copy()))
        if abs(ll_new - ll) < tol:
            break
        ll = ll_new
    return hist


데이터: 150개 점, 전체 평균 = [3.    2.163]
(본문의 퇴화 해 중심이 (2.98, 2.04) 근처였음 — 이 데이터의 평균과 같은 위치.)


## 3. EM으로 학습 (좋은 초기화)

진짜 중심 근처에서 초기화한 "운 좋은" 실행 — 6절의 무작위 초기화 30회와
비교할 기준이 된다. 세 \\(\mu_k\\)가 각 블롭 중심으로 수렴하고, \\(\Sigma_k\\)
고유값이 각 블롭의 "형상"(세로/가로로 길쭉함, 원형)을 잡았는지 확인한다.


In [3]:
# 좋은 초기화: 진짜 중심 근처 + 중간 크기 공분산
mus0 = centers + rng.randn(3, 2) * 0.2
covs0 = [np.eye(2) * 0.5 for _ in range(3)]
pis0 = [1 / 3, 1 / 3, 1 / 3]
hist = fit_gmm(X, mus0, covs0, pis0)
it, ll, mus, covs, pis = hist[-1]
good_ll = ll   # 6절의 무작위 초기화 실행들과 비교할 기준
print(f"반복 {it}에서 수렴, log-likelihood = {ll:.4f}   (6절 기준 값 good_ll)")
for k in range(3):
    e = np.sort(np.linalg.eigvalsh(covs[k]))[::-1]   # 큰 것부터
    print(f"  클러스터 {k + 1}: mu = {mus[k].round(3)}   Sigma 고유값 = {e.round(3)}   pi = {pis[k]:.3f}")
print("\n진짜 값:   mu =", [c.round(2) for c in centers])
print("           Sigma 고유값 = ", [np.sort(np.linalg.eigvalsh(c))[::-1].round(3) for c in covs_true])


반복 8에서 수렴, log-likelihood = -332.4395   (6절 기준 값 good_ll)
  클러스터 1: mu = [0.979 1.419]   Sigma 고유값 = [0.246 0.085]   pi = 0.333
  클러스터 2: mu = [3.959 4.033]   Sigma 고유값 = [0.317 0.19 ]   pi = 0.340
  클러스터 3: mu = [4.066 0.977]   Sigma 고유값 = [0.399 0.065]   pi = 0.327

진짜 값:   mu = [array([1. , 1.5]), array([4., 4.]), array([4., 1.])]
           Sigma 고유값 =  [array([0.36, 0.09]), array([0.25, 0.25]), array([0.3 , 0.09])]


## 4. 그림: 책임값 기반 색칠 + likelihood 등고선 (본문 그림 `ch15_gmm_contour.svg` 생성)

각 점을 책임값 \\(\gamma_{ik}\\) 가중치로 3색을 **섞어서** 색칠하면, 블롭
**경계 안쪽**의 점들이 두 색이 섞인 색으로 보인다 — k-means(점 하나에 딱
하나의 색)와의 가장 눈에 띄는 차이. 얇은 선은 학습된 GMM의 likelihood
등고선(타원호).


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Noto Sans CJK KR"
plt.rcParams["axes.unicode_minus"] = False

# 마지막 책임값 (n, 3)
lle = np.stack([np.log(max(pis[k], 1e-300)) + mvn_logpdf(X, mus[k], covs[k]) for k in range(3)])
m = lle.max(0, keepdims=True)
G = np.exp(lle - m).T
G /= G.sum(1, keepdims=True)                        # (n, 3) 책임값

# 색 = 3개 클러스터 색의 책임값 혼합 (경계 점들은 "혼합색")
cluster_colors = np.array([[0.12, 0.47, 0.71],   # 파랑
                           [0.84, 0.15, 0.16],   # 빨강
                           [0.17, 0.63, 0.17]])  # 초록
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.scatter(X[:, 0], X[:, 1], c=G @ cluster_colors, s=26, alpha=0.9, edgecolors="none")

# 학습된 GMM likelihood 등고선
xs = np.linspace(X[:, 0].min() - 0.6, X[:, 0].max() + 0.6, 240)
ys = np.linspace(X[:, 1].min() - 0.6, X[:, 1].max() + 0.6, 240)
XX, YY = np.meshgrid(xs, ys)
grid = np.column_stack([XX.ravel(), YY.ravel()])
lle_g = np.stack([np.log(max(pis[k], 1e-300)) + mvn_logpdf(grid, mus[k], covs[k]) for k in range(3)])
lg = lle_g.max(0, keepdims=True)
dens = np.exp(lle_g - lg).sum(0).reshape(XX.shape)
levels = np.logspace(np.log10(dens.max() * 1e-3), np.log10(dens.max() * 0.5), 8)
ax.contour(XX, YY, dens, levels=levels, colors="0.35", linewidths=0.9)

for k in range(3):
    ax.plot(mus[k][0], mus[k][1], "kx", ms=10, mew=2.2)
    ax.annotate(f"$\\mu_{{{k + 1}}}$", (mus[k][0], mus[k][1]),
                textcoords="offset points", xytext=(8, -14), fontsize=11)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("3 blobs (150 pts) + K=3 GMM (EM) - point color = responsibility mix, thin lines = fitted likelihood contours")
ax.set_aspect("equal")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig("ch15_gmm_contour.svg", bbox_inches="tight")
print("saved: ch15_gmm_contour.svg  (-> kor/src/images/ 에 복사해 본문에 삽입)")
plt.show()


saved: ch15_gmm_contour.svg  (-> kor/src/images/ 에 복사해 본문에 삽입)


## 5. \\(K\\)를 고르는 법: BIC 곡선 (본문 그림 `ch15_gmm_bic.svg` 생성)

likelihood로만 \\(K\\)를 고르면 늘 \\(K=m\\)를 고르게 된다(과적합, 본문 BIC
섹션). 대신 \\(\text{BIC} = -2\log\mathcal{L} + p\log m\\). \\(p\\)는 자유
모수 개수 — 전체 공분산을 쓰면 클러스터당 평균 2 + 공분산 3(2차원 대칭행렬
고유값 3개) = 5개, 그리고 혼합비율 \\(K-1\\)개를 더한다. 각 \\(K\\)마다
무작위 초기화 5회 돌려 그중 우도 최대인 것으로 BIC를 계산한다.


In [5]:
def fit_best(X, K, n_init=5, seed=0):
    # 무작위 초기화 n_init회, log-likelihood가 가장 높은 실행을 반환
    r = np.random.RandomState(seed)
    best = None
    for i in range(n_init):
        idx = r.choice(len(X), K, replace=False)    # K개 무작위 데이터 점이 초기 중심
        h = fit_gmm(X, X[idx].copy(),
                    [np.eye(2) * 0.5 for _ in range(K)], [1.0 / K] * K)
        if best is None or h[-1][1] > best:
            best = h[-1][1]
    return best

m = len(X)
print(f"{'K':>2}  {'log-likelihood':>16}  {'p (자유 모수)':>13}  {'BIC':>10}")
rows = []
for K in range(1, 5):
    ll = fit_best(X, K, n_init=5)
    p = K * 5 + (K - 1)          # 클러스터당 5개 + 혼합비율 (K-1)
    bic = -2 * ll + p * np.log(m)
    rows.append((K, ll, p, bic))
    print(f"{K:>2}  {ll:>16.2f}  {p:>13d}  {bic:>10.2f}")
K_best = rows[int(np.argmin([r[3] for r in rows]))][0]
print(f"\nBIC 최소: K = {K_best}")


 K    log-likelihood      p (자유 모수)         BIC
 1           -532.62              5     1090.29
 2           -405.45             11      866.01
 3           -332.44             17      750.06
 4           -319.59             23      754.42

BIC 최소: K = 3


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
Ks = [r[0] for r in rows]
lls = [r[1] for r in rows]
bics = [r[3] for r in rows]

ax = axes[0]
ax.plot(Ks, lls, "o-", color="#1f77b4", lw=1.5)
ax.set_xlabel("K (number of clusters)")
ax.set_ylabel("log-likelihood")
ax.set_title("(a) likelihood always improves as K grows - cannot be used to choose K")
ax.set_xticks(Ks)
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(Ks, bics, "o-", color="#d62728", lw=1.5)
ax.set_xlabel("K (number of clusters)")
ax.set_ylabel("BIC")
ax.set_title(f"(b) BIC = -2logL + p log m - minimum at K = {K_best}")
ax.set_xticks(Ks)
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig("ch15_gmm_bic.svg", bbox_inches="tight")
print("saved: ch15_gmm_bic.svg  (-> kor/src/images/ 에 복사해 본문에 삽입)")
plt.show()


saved: ch15_gmm_bic.svg  (-> kor/src/images/ 에 복사해 본문에 삽입)


## 6. 자주 하는 실수 1: 무작위 초기화 30회 — 지역최적해와 분산 붕괴

초기 중심 3개를 데이터 점에서 무작위로 고른다. 3개 클러스터에 초기값 3개이므로
"각 블롭마다 하나씩"인 확률은 \\(6/27 \approx 22\%\\)뿐 — 나머지는 어떤 블롭이
모델링되지 않거나 중심이 겹쳐지는, 더 나쁜 지역최적해에 갇힌다(본문 "자주 하는
실수 1"). 30회 실행의 log-likelihood 분포를 보고, 나쁜 해의 파라미터(중심
중첩/분산 붕괴)를 직접 살펴본다.


In [7]:
# good_ll은 3절에서 저장됨 (5절 BIC 루프가 ll 변수를 덮어쓰므로 이 자리에서 재할당하면 안 됨)

results = []
for seed in range(30):
    r = np.random.RandomState(seed)
    idx = r.choice(len(X), 3, replace=False)
    h = fit_gmm(X, X[idx].copy(),
                [np.eye(2) * 0.5 for _ in range(3)], [1 / 3] * 3)
    it, l, mu, co, pi = h[-1]
    dmin = min(np.linalg.norm(mu[i] - mu[j]) for i in range(3) for j in range(i + 1, 3))
    emin = min(float(np.linalg.eigvalsh(c).min()) for c in co)
    results.append(dict(seed=seed, ll=l, dmin=dmin, emin=emin, mus=mu, covs=co, pis=pi))

good = [r_ for r_ in results if r_["ll"] >= good_ll - 1e-3]
bad = [r_ for r_ in results if r_["ll"] < good_ll - 1e-3]
print(f"좋은 초기화(3절) logL = {good_ll:.2f}")
print(f"무작위 초기화 30회: 좋은 해 {len(good)}회, 더 나쁜 지역최적해 {len(bad)}회")
print(f"logL 범위: {min(r_['ll'] for r_ in results):.2f} ~ {max(r_['ll'] for r_ in results):.2f}")

if bad:
    w = min(bad, key=lambda r_: r_["ll"])
    print(f"\n나쁜 지역최적해 예시 (seed {w['seed']}): logL = {w['ll']:.2f} "
          f"(좋은 해와 차이 {good_ll - w['ll']:.2f})")
    for k in range(3):
        e = np.sort(np.linalg.eigvalsh(w["covs"][k]))[::-1]
        print(f"  클러스터 {k + 1}: mu = {w['mus'][k].round(3)}  "
              f"Sigma 고유값 = {e.round(4)}  pi = {w['pis'][k]:.3f}")
    print("  -> 중심이 겹치거나/몇 블롭이 모델링되지 못함 (본문 '자주 하는 실수 1'과 같은 구조)")
else:
    print("\n(이번 시드들에서는 모두 좋은 해에 도달했음 — 시도 수/시드를 바꾸면 나쁜 해가 나타남)")


좋은 초기화(3절) logL = -332.44
무작위 초기화 30회: 좋은 해 26회, 더 나쁜 지역최적해 4회
logL 범위: -403.08 ~ -332.44

나쁜 지역최적해 예시 (seed 1): logL = -403.08 (좋은 해와 차이 70.64)
  클러스터 1: mu = [2.524 1.209]  Sigma 고유값 = [2.6721 0.1686]  pi = 0.667
  클러스터 2: mu = [3.833 3.907]  Sigma 고유값 = [0.1918 0.1183]  pi = 0.244
  클러스터 3: mu = [4.278 4.516]  Sigma 고유값 = [0.3088 0.0826]  pi = 0.089
  -> 중심이 겹치거나/몇 블롭이 모델링되지 못함 (본문 '자주 하는 실수 1'과 같은 구조)


## 7. GMM 잠재변수 다이어그램 (본문 그림 `ch15_gmm_latent_var.svg` 생성)

Graphviz로 그리는 구조도 — 마름모 = 모수(\\(\pi, \mu, \sigma^2\\)), 빈 원 =
관측되지 않는 잠재변수 \\(z_i\\)(어느 클러스터인지), 채워진 원 = 관측 데이터
\\(x_i\\). \\(\pi_k \to z_i\\)(주사위), \\(z_i\\)와 \\((\mu, \sigma^2) \to x_i\\)
(그 클러스터의 정규분포에서 샘플링).


In [8]:
import graphviz

dot = r'''
digraph gmm_latent_var {
    rankdir=LR;
    graph [fontname="Noto Sans CJK KR", labelloc=t,
           label="GMM generation - (1) roll the categorical die (pi) to draw z_i, (2) sample x_i from that cluster normal N(mu_z, sigma^2_z). z_i is unobserved in the data (latent).",
           labeljust=c, bgcolor="white"];
    node [fontname="Noto Sans CJK KR", fontsize=12.5, color="#495057", penwidth=1.4, margin="0.16,0.10"];
    edge [fontname="Noto Sans CJK KR", fontsize=10.5, color="#495057", penwidth=1.6];

    pi  [label="\u03c0_k\n(mixing weight)", shape=diamond, fillcolor="#fff3cd", style=filled];
    mu  [label="\u03bc_k\n(mean)", shape=diamond, fillcolor="#fff3cd", style=filled];
    sig [label="\u03c3\xb2_k\n(variance)", shape=diamond, fillcolor="#fff3cd", style=filled];
    z   [label="z_i (which cluster)\nlatent - unobserved", shape=ellipse, fillcolor="white", style=filled];
    x   [label="x_i\nobserved data", shape=ellipse, fillcolor="#d1e7dd", style=filled];

    pi -> z  [label="①"];
    mu -> x;
    sig -> x;
    z -> x   [label="②"];
}
'''
g = graphviz.Source(dot)
g.render("ch15_gmm_latent_var", format="svg", cleanup=True)
print("saved: ch15_gmm_latent_var.svg  (-> kor/src/images/ 에 복사해 본문에 삽입)")


saved: ch15_gmm_latent_var.svg  (-> kor/src/images/ 에 복사해 본문에 삽입)
